In [1]:
import pandas as pd

In [2]:
from sqlalchemy import create_engine

db_user = "postgres"
db_pass = "postgres"
db_host = "localhost"
db_name = "crypto"

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}/{db_name}")

In [3]:
df = pd.read_sql("ALL_TICKER_DATA", engine)
copy_df = df.copy(deep = True)

In [4]:
remove_tickers = df[(df.open <= 10**-3) | (df.close <= 10**-3)].ticker.unique().tolist()
df = df[~ df.ticker.isin(remove_tickers)]

In [5]:
ticker_time_series = df.groupby('ticker').date.apply(lambda x: pd.date_range(start = x.min(), end = x.max()))
ticker_time_series = ticker_time_series.reset_index().explode('date')

In [6]:
df = ticker_time_series.merge(df, how = 'left', on = ['date', 'ticker'])

In [7]:
df = df.ffill(limit = 1).bfill(limit = 1)

In [8]:
df = df.drop(columns = ['open', 'high', 'low', 'close'])

In [10]:
df = df.assign(trend_adx_direction=lambda x: x['trend_adx_pos'] - x['trend_adx_neg']
         ).drop(columns = ['volume', "trend_macd", 'trend_macd_signal', 'trend_ema_fast', 'trend_ema_slow',
                          'volatility_bbl', 'volatility_bbh', 'volatility_bbm','daily_return',
                          'relative_strength_index', 'trend_adx_pos', 'trend_adx_neg','momentum_awesome_oscillator',
                          'momentum_stochastic_oscillator', 'momentum_stoch_signal', 'momentum_rate_of_change'])
df['trend_adx_direction'] = df['trend_adx_direction'].apply(lambda x: 1 if x > 0 else -1 if x < 0 else 0)

In [11]:
df.head()

,ticker,date,adjclose,volume_on_balance,volume_weighted_avg_price,volume_money_flow_index,volatility_bollinger_bands_width,volatility_bbp,volatility_bbhi,volatility_bbli,volatility_avg_true_range,trend_macd_diff,trend_avg_directional_index,trend_adx_direction
0,1INCH-USD,2020-12-25,2.328544,638225549.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
1,1INCH-USD,2020-12-26,1.596896,400572476.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
2,1INCH-USD,2020-12-27,1.062112,216594169.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
3,1INCH-USD,2020-12-28,1.110076,335931698.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
4,1INCH-USD,2020-12-29,0.887798,198014799.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0


In [12]:
df.shape

(281656, 14)

In [13]:
df.describe(percentiles=[.5])

,date,adjclose,volume_on_balance,volume_weighted_avg_price,volume_money_flow_index,volatility_bollinger_bands_width,volatility_bbp,volatility_bbhi,volatility_bbli,volatility_avg_true_range,trend_macd_diff,trend_avg_directional_index,trend_adx_direction
count,281656,281562.000000,2.815620e+05,279407.000000,279405.000000,278530.000000,278530.000000,281562.000000,281562.000000,281562.000000,276036.000000,281562.000000,281656.000000
mean,2022-06-25 21:39:22.950549760,1593.420082,2.220229e+08,1586.906543,52.564565,33.950962,0.495701,0.073494,0.050998,74.211261,-0.283420,28.174242,0.015902
min,2014-09-17 00:00:00,0.001038,-8.324242e+13,0.001387,0.000000,0.013428,-0.589334,0.000000,0.000000,0.000000,-1715.642032,0.000000,-1.000000
50%,2022-10-21 00:00:00,1.957985,1.677712e+09,2.023487,52.420644,26.942564,0.462300,0.000000,0.000000,0.143069,0.000060,25.696159,1.000000
max,2025-03-08 00:00:00,106146.265625,2.687295e+12,103557.008787,100.000000,1740.497368,1.589710,1.000000,1.000000,26504.764119,1882.855550,99.857243,1.000000
std,NaN,8456.161766,4.916403e+11,8413.160271,17.344456,30.649211,0.335934,0.260945,0.219993,418.037638,82.127740,13.174354,0.994947


In [14]:
df.columns

Index(['ticker', 'date', 'adjclose', 'volume_on_balance',
       'volume_weighted_avg_price', 'volume_money_flow_index',
       'volatility_bollinger_bands_width', 'volatility_bbp', 'volatility_bbhi',
       'volatility_bbli', 'volatility_avg_true_range', 'trend_macd_diff',
       'trend_avg_directional_index', 'trend_adx_direction'],
      dtype='object')

In [15]:
def rolling_max_normalize(series, window=100, buffer = 1e-6):
    rolling_max = series.abs().rolling(window, min_periods = 1).max() + buffer
    return series / rolling_max

def scale_percentage_columns(df):
    df['volume_money_flow_index'] = df['volume_money_flow_index'] / 100.0
    df['trend_avg_directional_index'] = df['trend_avg_directional_index'] / 100.0
    return df

def adaptive_normalize_features(df):

    columns_to_normalize = [
        'volume_on_balance',
        'volume_weighted_avg_price',
        'volatility_bollinger_bands_width',
        'volatility_avg_true_range',
        'trend_macd_diff'
    ]
    for column in columns_to_normalize:
        df[column] = rolling_max_normalize(df[column])
    return df

def preprocess_features(df):
    df = scale_percentage_columns(df)
    df = adaptive_normalize_features(df)
    return df

In [16]:
df = preprocess_features(df)

In [17]:
df.head()

,ticker,date,adjclose,volume_on_balance,volume_weighted_avg_price,volume_money_flow_index,volatility_bollinger_bands_width,volatility_bbp,volatility_bbhi,volatility_bbli,volatility_avg_true_range,trend_macd_diff,trend_avg_directional_index,trend_adx_direction
0,1INCH-USD,2020-12-25,2.328544,1.000000,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
1,1INCH-USD,2020-12-26,1.596896,0.627635,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
2,1INCH-USD,2020-12-27,1.062112,0.339369,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
3,1INCH-USD,2020-12-28,1.110076,0.526353,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
4,1INCH-USD,2020-12-29,0.887798,0.310258,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0


In [32]:
from sklearn.decomposition import PCA

In [34]:
df.columns

Index(['ticker', 'date', 'adjclose', 'volume_on_balance',
       'volume_weighted_avg_price', 'volume_money_flow_index',
       'volatility_bollinger_bands_width', 'volatility_bbp', 'volatility_bbhi',
       'volatility_bbli', 'volatility_avg_true_range', 'trend_macd_diff',
       'trend_avg_directional_index', 'trend_adx_direction'],
      dtype='object')

In [36]:
df.columns = df.columns.astype(str)

In [38]:
df.drop(columns = ['ticker','date', 'adjclose'])

,volume_on_balance,volume_weighted_avg_price,volume_money_flow_index,volatility_bollinger_bands_width,volatility_bbp,volatility_bbhi,volatility_bbli,volatility_avg_true_range,trend_macd_diff,trend_avg_directional_index,trend_adx_direction
0,1.000000,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,NaN,0.000000,0
1,0.627635,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,NaN,0.000000,0
2,0.339369,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,NaN,0.000000,0
3,0.526353,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,NaN,0.000000,0
4,0.310258,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,NaN,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...
281651,0.995546,0.426454,0.380597,0.317813,0.023991,0.0,0.0,0.290721,0.026074,0.281674,-1
281652,0.997056,0.423656,0.436747,0.326900,0.207616,0.0,0.0,0.278730,0.034479,0.280231,-1
281653,0.995661,0.419794,0.376497,0.340628,0.116812,0.0,0.0,0.272461,0.022823,0.275452,-1
281654,0.997285,0.414751,0.385172,0.353891,0.160421,0.0,0.0,0.269799,0.020850,0.274496,-1


In [47]:
pca = PCA(random_state = 1008, )
pca.fit(df.drop(columns = ['ticker','date', 'adjclose']).dropna().values)

PCA(random_state=1008)

In [48]:
pca.explained_variance_ratio_.cumsum()

array([0.56273143, 0.75073184, 0.8403697 , 0.89350892, 0.92290995,
       0.94898905, 0.97068503, 0.98012623, 0.98734474, 0.99400432,
       1.        ])

In [81]:
import gym
from gym import spaces
import numpy as np
import pandas as pd

class MultiStockEnv(gym.Env):
    def __init__(self, df, model, initial_balance=10000, lookback_window=10): #add model to the init.
        super(MultiStockEnv, self).__init__()
        self.df = df
        self.tickers = df['ticker'].unique()[:10]
        self.num_tickers = len(self.tickers)
        self.feature_columns = [col for col in df.columns if col not in ['date', 'ticker', 'adjclose']]
        self.num_features = len(self.feature_columns)
        self.action_space = spaces.Box(low=0, high=1, shape=(self.num_tickers,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.num_tickers, self.num_features + self.num_tickers), dtype=np.float32)
        self.initial_balance = initial_balance
        self.balance = initial_balance
        self.current_step = lookback_window
        self.max_steps = df['date'].nunique() - 1
        self.portfolio_value = initial_balance
        self.weights = np.array([1 / self.num_tickers] * self.num_tickers)
        self.lookback_window = lookback_window
        self.history = []
        self.model = model
        
    def _get_observation(self, current_date):
        observations = []
        for ticker in self.tickers:
            stock_data = self.df[(self.df['ticker'] == ticker) & (self.df['date'] == current_date)][self.feature_columns].values
            if len(stock_data) > 0:
                observations.append(stock_data[0])
            else:
                observations.append(np.zeros(self.num_features))  # Handle missing data
        observations = np.array(observations)
        observations = np.nan_to_num(observations, nan = 0.0)
        return np.concatenate((observations, np.tile(self.weights, (self.num_tickers, 1))), axis = 1)

    def _calculate_portfolio_value(self, current_date):
        portfolio_value = 0
        for i, ticker in enumerate(self.tickers):
            filtered_data = self.df[(self.df['ticker'] == ticker) & (self.df['date'] == current_date)]
            if not filtered_data.empty:
                price = filtered_data['adjclose'].values[0]
                portfolio_value += self.weights[i] * price
            else:
                previous_date = self.df[self.df['date'] < current_date].sort_values('date', ascending=False).groupby('ticker').head(1)
                if not previous_date.empty:
                    try:
                        price = previous_date[previous_date['ticker'] == ticker]['adjclose'].values[0]
                        portfolio_value += self.weights[i] * price
                    except IndexError:
                        portfolio_value += 0 # add zero.
                else:
                    portfolio_value += 0

        final_portfolio_value = portfolio_value * self.balance
        if np.isnan(final_portfolio_value):
            final_portfolio_value = 0 # add zero.
        return final_portfolio_value

    def _calculate_reward(self, current_date, previous_portfolio_value):
        current_portfolio_value = self._calculate_portfolio_value(current_date)
        initial_threshold = self.initial_balance * 1.2  # 20% above initial balance

        if current_portfolio_value >= initial_threshold:
            reward = (current_portfolio_value - initial_threshold) / self.initial_balance # smooth reward
        else:
            reward = (current_portfolio_value - previous_portfolio_value) / self.initial_balance * 10 # high penalty.
            
        if np.isnan(reward):
            reward = 0
        return reward
    
    def step(self, actions):
        previous_portfolio_value = self._calculate_portfolio_value(self.df['date'].unique()[self.current_step])
        actions, _states = model.predict(self._get_observation(self.df['date'].unique()[self.current_step])) # add this line
        actions = actions / np.sum(actions) #normalize actions.
        self.weights = actions
        current_date = self.df['date'].unique()[self.current_step]
        next_date = self.df['date'].unique()[min(self.current_step + 1, self.max_steps)]

        reward = self._calculate_reward(next_date, previous_portfolio_value)
        self.portfolio_value = self._calculate_portfolio_value(next_date)
        self.history.append((self._get_observation(current_date), actions, reward))

        print(f"Date: {next_date}, Portfolio Value: {self.portfolio_value}, Reward: {reward}") # Added print statement
        print(dict(zip(self.tickers, self.weights)))
        print()
        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self._get_observation(next_date), reward, done, {}

    def reset(self):
        self.weights = np.array([1 / self.num_tickers] * self.num_tickers)
        self.current_step = self.lookback_window
        return self._get_observation(self.df['date'].unique()[self.current_step])

In [82]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Assuming your dataframe is called 'df'
model = PPO("MlpPolicy", env, verbose=1)  # Create the model instance first.
env = MultiStockEnv(df, model) # pass the model.
env = DummyVecEnv([lambda: env])

model.learn(total_timesteps=100000)

# Example Usage
obs = env.reset()
done = False
while not done:
    action, _states = model.predict(obs) # this is not needed here.
    obs, rewards, done, info = env.step(action)
    env.render()

Using cpu device


C:\Users\ajayd\Home\Assignments\EAI 6980\Crypto-Market-Analysis\crypto\Lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Date: 2021-01-05 00:00:00, Portfolio Value: 4330.448768215167, Reward: -0.017187180908166733
{'1INCH-USD': 0.116691984, 'ACH-USD': 0.0, 'ADA-USD': 0.0, 'AERO29270-USD': 0.21781361, 'AIOZ-USD': 0.10195613, 'AKT-USD': 0.3377239, 'ALGO-USD': 0.0, 'AMP-USD': 0.15457992, 'APE18876-USD': 0.024199784, 'APT21794-USD': 0.04703467}

Date: 2021-01-06 00:00:00, Portfolio Value: 6571.246308631604, Reward: -0.012711896508813936
{'1INCH-USD': 0.3664118, 'ACH-USD': 0.0, 'ADA-USD': 0.0, 'AERO29270-USD': 0.3664118, 'AIOZ-USD': 0.1085373, 'AKT-USD': 0.06319234, 'ALGO-USD': 0.09544672, 'AMP-USD': 0.0, 'APE18876-USD': 0.0, 'APT21794-USD': 0.0}

Date: 2021-01-07 00:00:00, Portfolio Value: 1971.6238122735374, Reward: -0.8513997932307631
{'1INCH-USD': 0.0, 'ACH-USD': 0.0, 'ADA-USD': 0.40039104, 'AERO29270-USD': 0.45180956, 'AIOZ-USD': 0.0, 'AKT-USD': 0.022062164, 'ALGO-USD': 0.1257373, 'AMP-USD': 0.0, 'APE18876-USD': 0.0, 'APT21794-USD': 0.0}

Date: 2021-01-08 00:00:00, Portfolio Value: 1552.1895511167793, Re

KeyboardInterrupt: 

In [99]:
from datetime import datetime, timedelta

In [102]:
a = pd.DataFrame(columns = ['value', 'value2'])

,0,1
2025-03-26 00:01:48.910746,1,2


In [120]:
df.head()

,ticker,date,adjclose,volume_on_balance,volume_weighted_avg_price,volume_money_flow_index,volatility_bollinger_bands_width,volatility_bbp,volatility_bbhi,volatility_bbli,volatility_avg_true_range,trend_macd_diff,trend_avg_directional_index,trend_adx_direction
0,1INCH-USD,2020-12-25,2.328544,1.000000,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
1,1INCH-USD,2020-12-26,1.596896,0.627635,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
2,1INCH-USD,2020-12-27,1.062112,0.339369,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
3,1INCH-USD,2020-12-28,1.110076,0.526353,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0
4,1INCH-USD,2020-12-29,0.887798,0.310258,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0


In [129]:
TICKER_DATA = df.set_index(['date', 'ticker'])['adjclose']

class Portfolio:
    def __init__(self, initial_balance: float, account_open_date: datetime, asset_allocation: dict={}):
        self.asset_allocation = asset_allocation
        self.initial_balance = initial_balance
        self.invested_value = 0
        self.available_amount = initial_balance
        self.account_open_date = account_open_date
        self.snapshot_date = self.account_open_date
        self.previous_portfolio_value = self.initial_balance
        self.current_portfolio_value = self._get_portfolio_value()
        self.portfolio_snapshot = pd.DataFrame(columns = ['prev_PV', 'current_PV', 'asset_allocation_EOD', 
                                                          'invested_amount', 'available_amount', 'reward', 
                                                          'daily_transactions'])
        self.daily_transactions = 0
        self.current_reward = (self.snapshot_date, 0)
        
    def _get_portfolio_value(self):
        value = self.available_amount
        for ticker, weight in asset_allocation.items():
            value += invested_value * weight * TICKER_DATA.loc[(self.snapshot_date, ticker), 'adjclose']
        return value
    
    def update_allocation_and_date(self, new_asset_allocation: dict):
        self.daily_transactions = 0
        for ticker, weight in new_asset_allocation.items():
            if self.asset_allocation.get(ticker, 0) == weight:
                continue
            trade_price = (weight - asset_allocation[ticker]) * TICKER_DATA.loc[(self.snapshot_date, ticker), 'adjclose']
            self.available_amount -= trade_price
            self.invested_value += trade_price
            self.asset_allocation[ticker] = weight
            self.daily_transactions += 1
        self.previous_portfolio_value = self.current_portfolio_value
        self.current_portfolio_value = self._get_portfolio_value()
        self.add_portfolio_snapshot()
        self.snapshot_date += timedelta(days = 1)
    
    def add_portfolio_snapshot(self):
        today_df = pd.DataFrame([[
                                    self.previous_portfolio_value,
                                    self.current_portfolio_value,
                                    self.asset_allocation,
                                    self.invested_amount,
                                    self.available_amount,
                                    self._get_reward(),
                                    self.daily_transactions
                                    ]
                                ], 
                            columns = ['prev_PV', 'current_PV', 'asset_allocation_EOD', 'invested_amount', 
                                       'available_amount', 'reward', 'daily_transactions'], 
                            index = [self.snapshot_date])
        self.portfolio_snapshot = pd.concat([self.portfolio_snapshot, today_df])
    
    def _get_reward(self):
        if self.snapshot_date == self.current_reward[0]:
            return self.current_reward[1]
        
        if self.available_amount <= 0.1 * self.initial_balance:
            return -100
        
        reward = 0
        if len(self.asset_allocation) > 0:
            reward = -self.daily_transactions / len(self.asset_allocation)
        
        prev_ = self.previous_portfolio_value
        if prev_ == 0:
            prev_ = self.initial_balance
        
        portfolio_change = (self.current_portfolio_value - prev_) / prev_
        portfolio_change = np.log1p(portfolio_change) if portfolio_change > 0 else -np.expm1(-portfolio_change) * 10
        reward += portfolio_change
        
        if self.current_portfolio_value > self.initial_balance * 1.2: reward += 5
        elif self.current_portfolio_value < 0.8 * self.initial_balance: reward -= 5
        
        reward += self.available_amount / self.initial_balance
        
        recent_returns = self.portfolio_snapshot['current_PV'].tail(10).pct_change().dropna()
        if (recent_returns > 0).all():
            reward += 1
        
        if self.portfolio_snapshot.tail(10).pct_change().sum() > .2:
            reward += 5
        
        self.current_reward = (self.snapshot_date, reward)
        
        return reward